In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def plot_expert_classification(scn, mask_list, figsize=(20, 20), show_rgb=True):
    """
    Generates a 3x3 visualization grid.
    Computes the RGB composite only if show_rgb is True to optimize performance.
    """

    # Prepare the list of items to display
    # Create a copy to avoid modifying the original list outside the function
    items_to_plot = mask_list.copy()

    # RGB Calculation (True Color / Synthetic Green)
    if show_rgb:
        print("Calculating True Color composite (Synthetic Green)...")

        # Normalization function for 0-1 Reflectance
        def get_norm(band):
            data = np.nan_to_num(scn[band].values)
            if np.max(data) > 2.0: 
                data /= 100.0 # Convert percentage to 0-1 range
            return np.clip(data, 0, 1)

        # Retrieve raw components
        R = get_norm('C02')   # Red band
        NIR = get_norm('C03') # Near-Infrared (Veggie) band
        B = get_norm('C01')   # Blue band

        # Synthetic Green Formula (NOAA / CIMSS)
        # Used to simulate a green band for satellites lacking a dedicated green sensor.
        # Vegetation response (NIR) is mixed with Red and Blue to simulate chlorophyll.
        G = 0.45 * R + 0.10 * NIR + 0.45 * B
        G = np.clip(G, 0, 1)

        # Stack R, G, B channels
        rgb_combined = np.dstack([R, G, B])

        # Gamma Correction
        # Linear raw data appears too dark for human perception.
        # Applying a square root (approx. Gamma 2.2) brightens the image.
        rgb_img = np.clip(np.sqrt(rgb_combined), 0, 1)

        # Add the RGB image to the beginning of the plot list
        items_to_plot.insert(0, ("True Color (Synthetic Green)", rgb_img))

    # Retrieve projection and geographical extent
    area = scn['C02'].attrs['area']
    crs = area.to_cartopy_crs()
    extent = [
        area.area_extent[0], area.area_extent[2],
        area.area_extent[1], area.area_extent[3]
    ]

    # Setup Figure and Axes
    fig, axes = plt.subplots(3, 3, figsize=figsize, subplot_kw={'projection': crs})
    axes = axes.flatten()

    # Visualization Loop
    for i, (name, data) in enumerate(items_to_plot):
        if i >= len(axes): 
            break

        ax = axes[i]

        # Handle RGB (3 channels) vs. Masks (1 channel)
        if "RGB" in name or "Color" in name:
            ax.imshow(data, extent=extent, transform=crs, origin='upper')
        else:
            # Display masks using a grayscale colormap
            ax.imshow(data, extent=extent, transform=crs, origin='upper',
                      cmap='gray', vmin=0, vmax=1)

        # Add geographical features
        ax.add_feature(cfeature.COASTLINE, edgecolor='cyan', linewidth=1.5)
        ax.add_feature(cfeature.BORDERS, edgecolor='cyan', linestyle=':', linewidth=1)
        ax.set_title(name, fontsize=12, fontweight='bold')

        # Add gridlines
        gl = ax.gridlines(draw_labels=True, linewidth=0.2, color='white', alpha=0.5)
        gl.top_labels = gl.right_labels = False

    # Remove empty subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_model_performance(y_true, y_pred):
    """
    Computes and displays multiple metrics for semantic segmentation.
    y_true and y_pred should be ground truth and prediction tensors (e.g., shape (N, 128, 128)).
    """
    
    # Flatten predictions and ground truth labels into 1D vectors
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    # Class names as defined in the expert mask
    class_names = [
        "Unknown", "Surface", "Cumulus", "Stratus", "Mid-Level", 
        "Cirrus", "Cirrostratus", "Snow/Ice", "Cumulonimbus"
    ]
    
    # 1. Global Pixel Accuracy
    # Represents the total percentage of pixels correctly classified.
    accuracy = accuracy_score(y_true_flat, y_pred_flat)
    
    # 2. Precision, Recall, and F1-Score
    # Using macro average to give equal weight to each class regardless of frequency.
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_flat, y_pred_flat, average='macro', zero_division=0
    )
    
    # 3. Manual mIoU (Mean Intersection over Union) Calculation
    present_classes = np.unique(y_true_flat)
    ious = []
    for cls in present_classes:
        intersection = np.sum((y_true_flat == cls) & (y_pred_flat == cls))
        union = np.sum((y_true_flat == cls) | (y_pred_flat == cls))
        if union > 0:
            ious.append(intersection / union)
    miou = np.mean(ious)
    
    # Display Global Metrics
    print("GLOBAL EVALUATION REPORT")
    print(f"Global Pixel Accuracy : {accuracy:.4f}")
    print(f"Mean IoU (mIoU)       : {miou:.4f}")
    print(f"Macro Precision       : {precision:.4f}")
    print(f"Macro Recall          : {recall:.4f}")
    print(f"Macro F1-Score        : {f1:.4f}\n")
    
    # 4. Detailed Per-Class Report
    print("Per-Class Details (Precision, Recall, F1-Score)")
    
    # Filter class names to include only those present in the current dataset
    present_names = [class_names[i] for i in present_classes if i < len(class_names)]
    
    print(classification_report(
        y_true_flat, 
        y_pred_flat, 
        labels=present_classes,
        target_names=present_names, 
        zero_division=0
    ))
    
    return miou, accuracy, precision, recall, f1

In [ ]:
import os
import s3fs
import numpy as np
import gc
import time
from datetime import datetime
from satpy import Scene
from pyresample import create_area_def
from pyproj import CRS, Transformer
from sklearn.model_selection import train_test_split
from skimage.util import view_as_windows
from tqdm import tqdm

# ==========================================
# 1. UTILITY FUNCTIONS (ENGINE)
# ==========================================

def get_area_definition(zone_geo, resolution=2000):
    """
    Defines the target projection grid (map) for resampling.
    Converts geographic coordinates to the target Equidistant Cylindrical projection.
    """
    proj_dict = {'proj': 'eqc', 'datum': 'WGS84', 'lat_ts': 0, 'lon_0': 0}
    crs_src = CRS.from_epsg(4326)
    crs_dst = CRS.from_dict(proj_dict)
    transformer = Transformer.from_crs(crs_src, crs_dst, always_xy=True)

    xmin, ymin = transformer.transform(zone_geo[0], zone_geo[1])
    xmax, ymax = transformer.transform(zone_geo[2], zone_geo[3])

    area_def = create_area_def(
        area_id='custom_zone',
        projection=proj_dict,
        area_extent=[xmin, ymin, xmax, ymax],
        resolution=resolution,
        units='meters'
    )
    return area_def

def load_sat_data(target_date, bands_list):
    """
    Downloads and loads raw satellite bands into memory from AWS S3.
    Organizes files into local directories based on timestamp.
    """
    fs = s3fs.S3FileSystem(anon=True)
    year, day_of_year, hour = target_date.strftime('%Y'), target_date.strftime('%j'), target_date.strftime('%H')
    save_dir = os.path.join("/content/goes_images", target_date.strftime("%Y-%m-%d_%Hh%M"))
    os.makedirs(save_dir, exist_ok=True)

    # Search for files on S3 matching the instrument (ABI) and scan mode (M6)
    files_s3 = fs.glob(f"noaa-goes16/ABI-L1b-RadF/{year}/{day_of_year}/{hour}/*")
    pattern = target_date.strftime('s%Y%j%H00')

    downloads = []
    local_files = []

    for remote in files_s3:
        if any(f"M6{b}" in remote for b in bands_list) and pattern in remote:
            local = os.path.join(save_dir, remote.split('/')[-1])
            local_files.append(local)
            if not os.path.exists(local):
                downloads.append(remote)

    if downloads:
        try:
            fs.get(downloads, [os.path.join(save_dir, r.split('/')[-1]) for r in downloads])
        except Exception as e:
            print(f"Download error (non-critical if files exist): {e}")

    scn = Scene(filenames=local_files, reader='abi_l1b')
    scn.load(bands_list)
    return scn

def resample_with_cache(scn, area_def, cache_folder="./cache_geometry"):
    """
    Projects data onto the target grid using KD-Tree resampling with disk caching.
    """
    os.makedirs(cache_folder, exist_ok=True)
    new_scn = scn.resample(area_def, resampler='nearest', radius_of_influence=4000, cache_dir=cache_folder)
    return new_scn

def build_X_y(scn, mask_function):
    """
    Constructs the feature tensor (X) and label tensor (Y).
    X contains 12 normalized spectral bands.
    Y contains class indices derived from expert masks.
    """
    # 1. Compute masks using external expert function
    masks = mask_function(scn)

    # 2. Feature Extraction (X)
    bands = ['C01', 'C02', 'C03', 'C04', 'C05', 'C06',
             'C07', 'C08', 'C09', 'C10', 'C13', 'C15']
    shape = scn['C01'].shape
    X = np.empty((shape[0], shape[1], 12), dtype=np.float32)

    for i, b in enumerate(bands):
        data = scn[b].values.astype(np.float32)
        data = np.nan_to_num(data)
        if i <= 5: # Normalize reflectance bands (0-1 range)
            if np.max(data) > 2.0: 
                data /= 100.0
            data = np.clip(data, 0, 1)
        X[:, :, i] = data

    # 3. Label Mapping (Y)
    Y = np.zeros((shape[0], shape[1]), dtype=np.uint8)
    class_map = {
        "Surface": 1, "Cumulus": 2, "Stratus": 3, "Mid-Level": 4,
        "Cirrus": 5, "Cirrostratus": 6, "Snow/Ice": 7, "Cumulonimbus": 8
    }

    for name, mask in masks:
        for key, idx in class_map.items():
            if key in name:
                Y[mask] = idx
                break
    return X, Y

# ==========================================
# 2. MAIN PROCESSING FUNCTION (ORCHESTRATION)
# ==========================================

def generate_final_dataset(dates, zone, expert_func, patch_size=128, output_path="./dataset_final"):
    """
    Main pipeline: downloads, resamples, patches, and saves the dataset.
    """
    required_bands = ['C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C13', 'C15']
    area_def = get_area_definition(zone)

    all_X_patches = []
    all_Y_patches = []

    print(f"Starting pipeline for {len(dates)} timestamp(s)...")

    for target_date in tqdm(dates, desc="Processing dates"):
        try:
            # A. Load & Resample
            scn_raw = load_sat_data(target_date, required_bands)
            scn_proj = resample_with_cache(scn_raw, area_def)

            # B. Build Feature/Label Images
            X_img, Y_img = build_X_y(scn_proj, expert_func)

            # C. Vectorized Patch Extraction
            # Extracts sub-images of size (patch_size, patch_size)
            patches_X = view_as_windows(X_img, (patch_size, patch_size, 12), step=patch_size)
            patches_X = patches_X.reshape(-1, patch_size, patch_size, 12)

            patches_Y = view_as_windows(Y_img, (patch_size, patch_size), step=patch_size)
            patches_Y = patches_Y.reshape(-1, patch_size, patch_size)

            # D. Filtering
            # Removes patches that are empty or consist of missing data using the thermal band (C10)
            means_c10 = patches_X[..., 9].mean(axis=(1, 2))
            valid_mask = means_c10 > 0

            X_valid = patches_X[valid_mask]
            y_valid = patches_Y[valid_mask]

            if len(X_valid) > 0:
                all_X_patches.append(X_valid)
                all_Y_patches.append(y_valid)
                tqdm.write(f"Success {target_date}: {len(X_valid)} patches extracted.")
            else:
                tqdm.write(f"Warning {target_date}: No valid patches found (potentially empty image).")

            # Memory Management
            del scn_raw, scn_proj, X_img, Y_img, patches_X, patches_Y
            gc.collect()

        except Exception as e:
            tqdm.write(f"Error processing {target_date}: {e}")
            continue

    # ==========================================
    # 3. DATASET SPLIT & STORAGE
    # ==========================================
    if len(all_X_patches) > 0:
        print("\nMerging and Saving Dataset...")
        X_final = np.concatenate(all_X_patches, axis=0)
        Y_final = np.concatenate(all_Y_patches, axis=0)

        print(f"Total Patches: {X_final.shape[0]}")

        # Dataset Split: Train (70%), Validation (15%), Test (15%)
        X_tv, X_test, y_tv, y_test = train_test_split(X_final, Y_final, test_size=0.15, random_state=42)
        X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.176, random_state=42)

        os.makedirs(output_path, exist_ok=True)

        np.save(f"{output_path}/X_train.npy", X_train)
        np.save(f"{output_path}/y_train.npy", y_train)
        np.save(f"{output_path}/X_val.npy", X_val)
        np.save(f"{output_path}/y_val.npy", y_val)
        np.save(f"{output_path}/X_test.npy", X_test)
        np.save(f"{output_path}/y_test.npy", y_test)

        print(f"Data saved to {output_path}")
        print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
    else:
        print("No patches were generated.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from matplotlib.colors import ListedColormap, BoundaryNorm

# --- 1. Color Palette Configuration ---
# Hex codes corresponding to specific geophysical classifications
class_colors = [
    '#000000', # 0: Background/Empty
    '#8B4513', # 1: Surface
    '#87CEEB', # 2: Cumulus
    '#D3D3D3', # 3: Stratus
    '#A9A9A9', # 4: Mid-Level
    '#00FFFF', # 5: Cirrus
    '#FFFFFF', # 6: Cirrostratus
    '#0000FF', # 7: Snow/Ice
    '#FF00FF'  # 8: Cumulonimbus
]
cmap_expert = ListedColormap(class_colors)
norm_expert = BoundaryNorm(np.arange(-0.5, 9.5, 1), cmap_expert.N)

# --- 2. Main Visualization Function ---
def plot_rgb_patch(X_dataset, Y_dataset, index):
    """
    Displays a data patch using Synthetic True Color (RGB) reconstruction.
    """
    if index >= len(X_dataset):
        print(f"Error: Index {index} is out of bounds. The dataset contains {len(X_dataset)} patches.")
        return

    patch_X = X_dataset[index]
    patch_Y = Y_dataset[index]

    # ---------------------------------------------------------
    # SYNTHETIC RGB CALCULATION
    # Tensor X index mapping:
    # 0 = C01 (Blue) | 1 = C02 (Red) | 2 = C03 (Near-IR)
    # ---------------------------------------------------------
    B = patch_X[..., 0]
    R = patch_X[..., 1]
    NIR = patch_X[..., 2]

    # Synthetic Green Formula (NOAA/CIMSS)
    G = 0.45 * R + 0.10 * NIR + 0.45 * B
    G = np.clip(G, 0, 1) # Safety clipping

    # Stacking and Gamma Correction
    # Raw satellite data is linear; square root approximates Gamma 2.2 for display.
    rgb_img = np.dstack([R, G, B])
    rgb_img = np.clip(np.sqrt(rgb_img), 0, 1)

    # ---------------------------------------------------------
    # PLOTTING (1 row, 4 columns)
    # ---------------------------------------------------------
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f"Patch ID: {index}", fontsize=16, fontweight='bold')

    # Subplot 1: Synthetic RGB
    axes[0].imshow(rgb_img)
    axes[0].set_title("True Color (RGB)")
    axes[0].axis('off')

    # Subplot 2: Pure Visible Band (C02)
    im1 = axes[1].imshow(R, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title("Raw Visible (C02)")
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    # Subplot 3: Thermal Infrared Band (C13 - Index 10)
    im2 = axes[2].imshow(patch_X[..., 10], cmap='gray_r')
    axes[2].set_title("Thermal IR (C13)")
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    # Subplot 4: Expert Mask (Ground Truth)
    im3 = axes[3].imshow(patch_Y, cmap=cmap_expert, norm=norm_expert)
    axes[3].set_title("Expert Mask (Ground Truth)")
    axes[3].axis('off')

    # Configure Colorbar Legend
    cbar = plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04, ticks=range(9))
    cbar.ax.set_yticklabels([
        'Empty', 'Surface', 'Cumulus', 'Stratus', 'Mid-Level', 
        'Cirrus', 'Cirrostratus', 'Snow/Ice', 'Cumulonimbus'
    ])

    plt.tight_layout()
    plt.show()

# --- 3. Random Selection Helper ---
def plot_random_patch(X_dataset, Y_dataset):
    """ Selects a random patch from the dataset and visualizes it. """
    random_index = random.randint(0, len(X_dataset) - 1)
    plot_rgb_patch(X_dataset, Y_dataset, random_index)

In [ ]:
# ==========================================
# 5. PCA ANALYSIS AND BAND SELECTION
# ==========================================

# --- 5.0 IMPORTS AND CONFIGURATION ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from scipy.stats import f_oneway

# Path to the processed dataset
DATASET_PATH = "./dataset_multizone_final"

# Stratified pixel sample size to ensure RAM safety in environments like Google Colab
SAMPLE_SIZE = 50_000  

# Spectral band identifiers
BAND_NAMES = ['C01', 'C02', 'C03', 'C04', 'C05', 'C06',
              'C07', 'C08', 'C09', 'C10', 'C13', 'C15']

# Mapping of class indices to geophysical labels
CLASS_NAMES = {
    0: 'Background', 
    1: 'Surface', 
    2: 'Cumulus', 
    3: 'Stratus', 
    4: 'Mid-Level',
    5: 'Cirrus', 
    6: 'Cirrostratus', 
    7: 'Snow/Ice', 
    8: 'Cumulonimbus'
}

# Color palette for visualization consistent with previous processing steps
PALETTE = {
    0: '#aaaaaa', 
    1: '#8B4513', 
    2: '#87CEEB', 
    3: '#D3D3D3', 
    4: '#A9A9A9',
    5: '#00FFFF', 
    6: '#FFFFFF', 
    7: '#0000FF', 
    8: '#FF00FF'
}

In [ ]:
# --- 5.1 LOADING AND STRATIFIED SUB-SAMPLING ---
print("Loading X_train / y_train...")
X_train = np.load(f"{DATASET_PATH}/X_train.npy")  # Shape: (N, 128, 128, 12)
y_train = np.load(f"{DATASET_PATH}/y_train.npy")  # Shape: (N, 128, 128)

# Flatten the spatial dimensions to treat each pixel as an individual observation
N, H, W, C = X_train.shape
X_flat = X_train.reshape(-1, C).astype(np.float32)
y_flat = y_train.reshape(-1).astype(np.uint8)

print(f"Total Pixels: {len(X_flat):,}")
print("Class Distribution:")
classes_uniq, counts = np.unique(y_flat, return_counts=True)
for c, n in zip(classes_uniq, counts):
    class_name = CLASS_NAMES.get(c, "Unknown")
    print(f"{class_name:>15s} (id={c}) : {n:>10,} px ({100*n/len(y_flat):.1f}%)")

# Stratified sub-sampling: 
# Selects an equal number of pixels per class to prevent bias from dominant classes.
# Excludes class 0 (Background) from the sampling process.
rng = np.random.default_rng(42)
active_classes = [c for c in classes_uniq if c > 0]
per_class_limit = SAMPLE_SIZE // len(active_classes)

sampled_indices = []
for cls in active_classes:
    idx_cls = np.where(y_flat == cls)[0]
    # Ensure we don't request more samples than available for the specific class
    sampled_indices.append(rng.choice(idx_cls, size=min(per_class_limit, len(idx_cls)), replace=False))

indices = np.concatenate(sampled_indices)
rng.shuffle(indices)

# Final sampled feature and label sets
X_samp = X_flat[indices]
y_samp = y_flat[indices]

print(f"\nSub-sample complete: {len(X_samp):,} pixels ({len(active_classes)} classes represented)")

In [ ]:
# --- 5.2 STANDARDIZATION ---
# Standardization is mandatory because the bands are heterogeneous:
#   - Visible Bands (C01-C06): Reflectance range [0.0 - 1.0]
#   - Thermal Bands (C07-C15): Brightness Temperature range [200K - 320K]
# Without StandardScaler, PCA would be heavily biased towards the thermal bands 
# due to their much larger numerical scales.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_samp)

print("Standardization complete.")
print(f"   Mean per band (should be ~0): {X_scaled.mean(axis=0).round(3)}")
print(f"   Std per band (should be ~1):  {X_scaled.std(axis=0).round(3)}")

In [ ]:
# --- 5.3 FULL PCA — VARIANCE AND LOADINGS ---
print("Calculating full PCA (12 components)...")
pca_full = PCA(n_components=12)
pca_full.fit(X_scaled)

# Retrieve explained variance and cumulative sum
var_exp = pca_full.explained_variance_ratio_
var_cumul = np.cumsum(var_exp)

# Initialize visualization (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("GOES-16 PCA Analysis — 12 Bands", fontsize=14, fontweight='bold')

# 1. Scree Plot
# Displays how much information (variance) is captured by each principal component.
axes[0].bar(range(1, 13), var_exp * 100, color='steelblue', alpha=0.8, label='Individual Variance')
axes[0].plot(range(1, 13), var_cumul * 100, 'o-', color='tomato', lw=2, label='Cumulative Variance')

# Horizontal lines for standard information thresholds (90% and 95%)
for threshold, ls in [(90, '--'), (95, ':')]:
    axes[0].axhline(threshold, color='gray', ls=ls, alpha=0.7, label=f'{threshold}% Threshold')

axes[0].set_xlabel("Principal Component (PC)")
axes[0].set_ylabel("Explained Variance (%)")
axes[0].set_title("Scree Plot")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. PCA Loadings for PC1 and PC2
# Visualizes the contribution of each spectral band to the first two components.
loadings = pca_full.components_[:2]
x_indices = np.arange(12)
bar_width = 0.35

axes[1].barh(x_indices + bar_width/2, loadings[0], height=bar_width, alpha=0.8, color='steelblue', label='PC1')
axes[1].barh(x_indices - bar_width/2, loadings[1], height=bar_width, alpha=0.8, color='tomato', label='PC2')

axes[1].set_yticks(x_indices)
axes[1].set_yticklabels(BAND_NAMES)
axes[1].set_title("PC1 & PC2 Loadings\n(Contribution of each band)")
axes[1].axvline(0, color='black', lw=0.8)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("pca_variance_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

# Calculate specific thresholds for dimension reduction
n_90 = int(np.argmax(var_cumul >= 0.90)) + 1
n_95 = int(np.argmax(var_cumul >= 0.95)) + 1

print("\nScree Plot Results:")
print(f"    - {n_90} components required to reach 90% variance")
print(f"    - {n_95} components required to reach 95% variance")

# Detailed breakdown per component
for i, (v, vc) in enumerate(zip(var_exp, var_cumul)):
    print(f"    PC{i+1:02d} : {v*100:5.1f}% (Cumulative: {vc*100:5.1f}%)")

In [ ]:
# --- 5.4 2D PCA VISUALIZATION — CLASS SEPARABILITY ---
# Reduce the 12 spectral dimensions to 2 principal components for visualization.
pca_2d = PCA(n_components=2)
X_pca2 = pca_2d.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(11, 8))

# Plot each class using the sampled data to evaluate how well spectral bands distinguish them.
for cls in active_classes:
    mask = y_samp == cls
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
               s=2, alpha=0.35, color=PALETTE[cls], label=CLASS_NAMES[cls])

# Label axes with the percentage of total variance captured by each component.
ax.set_xlabel(f"PC1 ({var_exp[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({var_exp[1]*100:.1f}% variance)")
ax.set_title("2D PCA — GOES-16 Cloud Class Separability\n"
             "(Well-separated clusters indicate strong spectral signatures for classification)")

# Configure legend and grid for clarity.
ax.legend(markerscale=6, framealpha=0.9, fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("pca_class_separability.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 5.5 INTER-BAND CORRELATION MATRIX ---
# Correlation is calculated on raw data (non-standardized) to observe 
# physical relationships between spectral bands.
correlation_matrix = np.corrcoef(X_samp.T)

fig, ax = plt.subplots(figsize=(9, 8))
# Using RdBu_r colormap (Red-Blue reversed) to emphasize strong positive/negative correlations
im = ax.imshow(correlation_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Pearson Correlation Coefficient')

# Axis configuration
ax.set_xticks(range(12))
ax.set_xticklabels(BAND_NAMES, rotation=45, ha='right')
ax.set_yticks(range(12))
ax.set_yticklabels(BAND_NAMES)
ax.set_title("GOES-16 Inter-Band Correlation\n"
             "(High red values indicate redundant bands — avoid using both in model features)")

# Annotate each cell with the correlation value
for i in range(12):
    for j in range(12):
        val = correlation_matrix[i, j]
        ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                fontsize=6.5,
                color='white' if abs(val) > 0.75 else 'black')

plt.tight_layout()
plt.savefig("band_correlation_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

# Detect and display highly correlated pairs
print("\nHighly correlated band pairs (|r| > 0.90) indicating redundancy:")
redundancy_found = False
for i in range(12):
    for j in range(i+1, 12):
        if abs(correlation_matrix[i, j]) > 0.90:
            print(f"   {BAND_NAMES[i]} <-> {BAND_NAMES[j]} (r = {correlation_matrix[i,j]:+.3f})")
            redundancy_found = True

if not redundancy_found:
    print("   No pairs found with correlation above 0.90.")

In [ ]:
# --- 5.6 F-SCORE (ANOVA) — DISCRIMINANT POWER PER BAND ---
# The F-Score measures the ratio of between-class variance to within-class variance.
# A higher F-Score indicates that the spectral band is more effective at 
# discriminating between the 8 defined cloud classes.

print("Calculating ANOVA F-Score per band...")
f_scores = []
for b in range(12):
    # Group pixel values by class for the current band
    groups = [X_samp[y_samp == cls, b] for cls in active_classes]
    f_stat, _ = f_oneway(*groups)
    f_scores.append(f_stat)

f_scores = np.array(f_scores)
# Sort indices from most discriminant to least discriminant
ranking_order = np.argsort(f_scores)[::-1] 

fig, ax = plt.subplots(figsize=(11, 5))

# Define a color scheme based on ranking performance
bar_colors = ['gold' if i < 4 else ('lightcoral' if i < 6 else 'steelblue')
              for i in range(12)]

# Plot the F-Scores in descending order
ax.bar([BAND_NAMES[i] for i in ranking_order],
       f_scores[ranking_order],
       color=[bar_colors[r] for r in range(12)],
       edgecolor='black', linewidth=0.5)

ax.set_title("ANOVA F-Score — Spectral Band Discriminant Power\n"
             "(Higher score = better separation of the 8 cloud classes)")
ax.set_ylabel("F-Score")
ax.set_xlabel("Spectral Band")
ax.grid(axis='y', alpha=0.35)

# Annotate bars with their numerical F-Score values
for patch, i_band in zip(ax.patches, ranking_order):
    ax.text(patch.get_x() + patch.get_width()/2,
            patch.get_height() * 1.01,
            f"{f_scores[i_band]:.0f}",
            ha='center', va='bottom', fontsize=8)

# Legend configuration
legend_items = [
    mpatches.Patch(color='gold',       label='Top 4 (Recommended)'),
    mpatches.Patch(color='lightcoral', label='Rank 5-6 (Optional)'),
    mpatches.Patch(color='steelblue',  label='Least Discriminant'),
]
ax.legend(handles=legend_items, framealpha=0.9)

plt.tight_layout()
plt.savefig("band_f_scores.png", dpi=150, bbox_inches='tight')
plt.show()

# Extract top performing bands for feature selection
top4_bands = [BAND_NAMES[i] for i in ranking_order[:4]]
top3_bands = [BAND_NAMES[i] for i in ranking_order[:3]]

print("\nComplete band ranking by F-Score:")
for rank, i in enumerate(ranking_order):
    # Use a simple text marker for the top 4 bands
    marker = "[TOP]" if rank < 4 else "     "
    print(f"   #{rank+1:02d} {marker} {BAND_NAMES[i]:<4s}  F = {f_scores[i]:>10,.1f}")

print(f"\n   Selection Top 3: {top3_bands}")
print(f"   Selection Top 4: {top4_bands}")

In [ ]:
# --- 5.7 LDA — MAXIMUM SEPARABILITY (DIAGNOSTIC BONUS) ---
# LDA (Linear Discriminant Analysis) identifies axes that MAXIMIZE class separation.
# It is significantly more effective than PCA for visualizing how well spectral 
# features distinguish between different categories.

print("LDA — Projecting into the maximally discriminant space...")
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_scaled, y_samp)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. LDA Projection Plot
for cls in active_classes:
    mask = y_samp == cls
    axes[0].scatter(X_lda[mask, 0], X_lda[mask, 1],
                    s=2, alpha=0.4, color=PALETTE[cls], label=CLASS_NAMES[cls])

axes[0].set_xlabel("LD1")
axes[0].set_ylabel("LD2")
axes[0].set_title("2D LDA — Maximum Separability\n(Compare with the 2D PCA above)")
axes[0].legend(markerscale=5, framealpha=0.9, fontsize=9)
axes[0].grid(alpha=0.2)

# 2. Comparative Analysis Summary (Text Box)
axes[1].axis('off')
summary_text = (
    "ANALYSIS SUMMARY\n"
    "===================================\n\n"
    f"PCA: {n_90} components -> 90% variance\n"
    f"     {n_95} components -> 95% variance\n\n"
    f"RECOMMENDED BANDS\n"
    f"   Top 3: {', '.join(top3_bands)}\n"
    f"   Top 4: {', '.join(top4_bands)}\n\n"
    f"REDUNDANT BANDS (r > 0.90):\n"
)

# Detect and add redundant pairs to the summary text
for i in range(12):
    for j in range(i+1, 12):
        if abs(correlation_matrix[i, j]) > 0.90:
            summary_text += f"   {BAND_NAMES[i]} <-> {BAND_NAMES[j]} (r={correlation_matrix[i,j]:.2f})\n"

axes[1].text(0.05, 0.95, summary_text,
             transform=axes[1].transAxes,
             fontsize=11, verticalalignment='top',
             fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig("lda_class_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nPCA/LDA Analysis complete. Results saved:")
print("  - pca_variance_analysis.png")
print("  - pca_class_separability.png")
print("  - band_correlation_matrix.png")
print("  - band_f_scores.png")
print("  - lda_class_analysis.png")

print(f"\nNext Step: Train U-Net using bands {top4_bands} (or {top3_bands} for a minimal configuration).")

In [ ]:
# --- 5.8 3D PCA VISUALIZATION (INTERACTIVE) ---
# Relevant here as PC3 (capturing ~4.2% of variance) helps distinguish ice clouds 
# (Cirrus, Cirrostratus) which often overlap when projected only onto the first two axes.

from mpl_toolkits.mplot3d import Axes3D  # noqa

# Perform PCA with 3 components
pca_3d = PCA(n_components=3)
X_pca3 = pca_3d.fit_transform(X_scaled)

var_ratio_3d = pca_3d.explained_variance_ratio_

# Initialize Matplotlib 3D Plot
fig = plt.figure(figsize=(13, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot each class in the 3D feature space
for cls in active_classes:
    mask = y_samp == cls
    ax.scatter(X_pca3[mask, 0],
               X_pca3[mask, 1],
               X_pca3[mask, 2],
               s=1, alpha=0.25,
               color=PALETTE[cls],
               label=CLASS_NAMES[cls])

# Configure axes and labels
ax.set_xlabel(f"PC1 ({var_ratio_3d[0]*100:.1f}%)", fontsize=10)
ax.set_ylabel(f"PC2 ({var_ratio_3d[1]*100:.1f}%)", fontsize=10)
ax.set_zlabel(f"PC3 ({var_ratio_3d[2]*100:.1f}%)", fontsize=10)
ax.set_title("3D PCA — GOES-16 Cloud Class Separability\n"
             "Rotate the plot to explore hidden clusters")
ax.legend(markerscale=6, fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig("pca_3d_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

# Interactive Plotly implementation for browser-based exploration
try:
    import plotly.express as px
    import pandas as pd

    # Prepare data for Plotly
    df_3d = pd.DataFrame({
        'PC1': X_pca3[:, 0],
        'PC2': X_pca3[:, 1],
        'PC3': X_pca3[:, 2],
        'Class': [CLASS_NAMES[c] for c in y_samp]
    })

    # Map the palette to class labels
    color_map = {CLASS_NAMES[k]: v for k, v in PALETTE.items()}

    # Create 3D Interactive Scatter Plot
    fig_px = px.scatter_3d(
        df_3d, x='PC1', y='PC2', z='PC3',
        color='Class',
        color_discrete_map=color_map,
        opacity=0.4,
        title=f"Interactive 3D PCA — Total Variance: {(sum(var_ratio_3d)*100):.1f}%"
    )
    
    # Adjust marker size for better visibility
    fig_px.update_traces(marker=dict(size=2))
    fig_px.show()
    print("Interactive Plotly graph generated successfully.")

except ImportError:
    print("Plotly not available. Install it using: !pip install plotly")

In [ ]:
# --- 5.9 FINAL SELECTION VALIDATION: C13, C15, C07, C04 ---
# Verification that these 4 selected bands have low inter-correlation 
# to ensure feature diversity for the model.

selected_bands = ['C13', 'C15', 'C07', 'C04']
selected_indices = [BAND_NAMES.index(b) for b in selected_bands]

print("=" * 55)
print("FINAL BAND SELECTION VALIDATION")
print("=" * 55)

print("\n1. ANOVA F-Score for each selected band:")
for b, i in zip(selected_bands, selected_indices):
    # Determine the rank in the original sorted F-score list
    rank = list(ranking_order).index(i) + 1
    print(f"   {b:<4s}  Rank #{rank:02d}  F = {f_scores[i]:>10,.1f}")

print("\n2. Cross-correlations (Ideally < 0.85):")
selected_corr = correlation_matrix[np.ix_(selected_indices, selected_indices)]
for i in range(len(selected_bands)):
    for j in range(i+1, len(selected_bands)):
        r_val = selected_corr[i, j]
        # Status marker: Warning if correlation is too high, OK otherwise
        status = "WARNING" if abs(r_val) > 0.85 else "OK"
        print(f"   [{status}] {selected_bands[i]} <-> {selected_bands[j]} : r = {r_val:+.3f}")

print("\n3. Physical role of each band:")
physical_roles = {
    'C13': "Clean IR Window (10.3um) -> Provides cloud-top temperature.",
    'C15': "Dirty IR Window (12.3um) -> C13-C15 Split Window Difference identifies thin ice clouds.",
    'C07': "Shortwave IR (3.9um) -> Nighttime convection, fires, and ice vs. water phase detection.",
    'C04': "Cirrus Band (1.37um) -> High-altitude sensitivity; identifies thin cirrus otherwise invisible."
}
for b in selected_bands:
    print(f"   {b} : {physical_roles[b]}")

print("\n4. Exclusion Logic: Why not C10 (despite Rank #4)?")
idx_c10 = BAND_NAMES.index('C10')
for ref_band in ['C08', 'C09', 'C15']:
    idx_ref = BAND_NAMES.index(ref_band)
    print(f"   C10 <-> {ref_band} : r = {correlation_matrix[idx_c10, idx_ref]:+.3f}")
print("   - Result: C10 (Lower-level Water Vapor) is highly redundant with C08/C09.")
print("   - Selection Strategy: C04 provides unique cirrus data that C10 cannot replicate.")

print("\n" + "=" * 55)
print("RECOMMENDED DATASET FOR U-NET")
print("=" * 55)
print(f"   Bands: {selected_bands} (4 channels)")
print(f"   Input Shape X: (N, 128, 128, 4) instead of (N, 128, 128, 12)")
print(f"   Memory Optimization: -67%")
print("   Next Step: Generate 4-band dataset and initialize U-Net training.")

In [ ]:
# ==========================================
# 6. GENERATION OF THE OPTIMAL BAND DATASET
# ==========================================

import numpy as np
import os

# Configuration of paths
DATASET_PATH = "./dataset_multizone_final"
DATASET_OUTPUT = "./dataset_multizone_final_optimized"
os.makedirs(DATASET_OUTPUT, exist_ok=True)

# Mapping of indices based on the original 12-band X tensor:
# BAND_NAMES = ['C01','C02','C03','C04','C05','C06','C07','C08','C09','C10','C13','C15']
# Indices      [ 0,    1,    2,    3,    4,    5,    6,    7,    8,    9,    10,   11 ]

# Selected bands based on PCA and F-Score analysis
SELECTED_BANDS = ['C13', 'C15', 'C07', 'C04']
SELECTED_INDICES = [10, 11, 6, 3] 

print("=" * 50)
print(f"Selected Bands: {SELECTED_BANDS}")
print(f"Indices in X:   {SELECTED_INDICES}")
print("=" * 50)

# Process each data split (train, validation, and test)
for split in ['train', 'val', 'test']:
    # Load original datasets (N, 128, 128, 12)
    X = np.load(f"{DATASET_PATH}/X_{split}.npy")
    y = np.load(f"{DATASET_PATH}/y_{split}.npy")

    # Slice the tensor to keep only the selected spectral channels
    X_selected = X[:, :, :, SELECTED_INDICES]

    # Save the optimized datasets
    np.save(f"{DATASET_OUTPUT}/X_{split}.npy", X_selected)
    np.save(f"{DATASET_OUTPUT}/y_{split}.npy", y)

    print(f"Split: {split:5s} -> X: {X_selected.shape}  y: {y.shape}")

# Final summary
num_selected = len(SELECTED_INDICES)
memory_gain = (1 - num_selected / 12) * 100

print(f"\nOptimized dataset saved to {DATASET_OUTPUT}")
print(f"Memory reduction: {memory_gain:.0f}%")

In [ ]:
DATASET_3B    = "./dataset_multizone_final_3bands"


In [ ]:
# ==========================================
# 7. U-NET — ARCHITECTURE & TRAINING
# ==========================================


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import os
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")

# ==========================================

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class GOESDataset(Dataset):
    def __init__(self, dataset_path, split, augment=False, stats=None):
        """
        PyTorch Dataset for GOES satellite imagery.
        Loads preprocessed patches and applies normalization and optional augmentation.
        """
        self.X = np.load(f"{dataset_path}/X_{split}.npy")
        self.y = np.load(f"{dataset_path}/y_{split}.npy")
        self.augment = augment

        # GLOBAL Normalization:
        # Statistics must be calculated on the training set only and reused for 
        # validation/test sets to prevent distribution leakage.
        if stats is None:
            # Calculate min/max values per channel for the current split (typically for training)
            self.vmin = self.X.reshape(-1, self.X.shape[-1]).min(axis=0)
            self.vmax = self.X.reshape(-1, self.X.shape[-1]).max(axis=0)
        else:
            # Apply training set statistics to ensure consistency across splits
            self.vmin, self.vmax = stats

        self.X = self._normalize(self.X)

    def _normalize(self, X):
        """
        Applies Min-Max scaling to bring pixel values into the [0, 1] range.
        """
        X = X.astype(np.float32)
        for c in range(X.shape[-1]):
            val_range = self.vmax[c] - self.vmin[c]
            if val_range > 0:
                X[:, :, :, c] = (X[:, :, :, c] - self.vmin[c]) / val_range
        return np.clip(X, 0, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Convert to torch tensor and permute dimensions from (H, W, C) to (C, H, W)
        x = torch.from_numpy(self.X[idx]).permute(2, 0, 1)
        y = torch.from_numpy(self.y[idx].astype(np.int64))

        # Data Augmentation: Flips and Rotations
        # Applied to both input (x) and mask (y) to maintain spatial correspondence
        if self.augment:
            # Random Horizontal Flip
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[2])
                y = torch.flip(y, dims=[1])
            
            # Random Vertical Flip
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[1])
                y = torch.flip(y, dims=[0])
            
            # Random 90-degree Rotation
            # Helps diversify highly correlated spatial patches
            k = torch.randint(0, 4, (1,)).item()
            x = torch.rot90(x, k, dims=[1, 2])
            y = torch.rot90(y, k, dims=[0, 1])

        return x, y

# Workflow for instantiation to prevent data leakage:

# 1. Training dataset calculates its own stats
train_ds = GOESDataset(DATASET_3B, 'train', augment=True)
train_stats = (train_ds.vmin, train_ds.vmax)

# 2. Validation and Test datasets reuse the training stats
val_ds  = GOESDataset(DATASET_3B, 'val',  augment=False, stats=train_stats)
test_ds = GOESDataset(DATASET_3B, 'test', augment=False, stats=train_stats)

# DataLoaders configuration
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        layers += [
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        self.block = nn.Sequential(*layers)

    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, n_classes=9, features=[32, 64, 128, 256]):
        super().__init__()

        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        ch = in_channels
        for f in features:
            # Dropout croissant avec la profondeur
            drop = 0.1 if f <= 64 else 0.4
            self.encoders.append(DoubleConv(ch, f, dropout=drop))
            self.pools.append(nn.MaxPool2d(2))
            ch = f

        # Bottleneck avec dropout fort
        self.bottleneck = DoubleConv(features[-1], features[-1]*2, dropout=0.5)

        self.upconvs  = nn.ModuleList()
        self.decoders = nn.ModuleList()
        ch = features[-1] * 2
        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(ch, f, kernel_size=2, stride=2))
            drop = 0.1 if f <= 64 else 0.3
            self.decoders.append(DoubleConv(f*2, f, dropout=drop))
            ch = f

        self.head = nn.Conv2d(features[0], n_classes, kernel_size=1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bottleneck(x)
        for upconv, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = upconv(x)
            if x.shape != skip.shape:
                x = nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = dec(x)
        return self.head(x)


model = UNet(in_channels=4, n_classes=9, features=[32, 64, 128, 256]).to(DEVICE)
dummy = torch.randn(2, 4, 128, 128).to(DEVICE)
out   = model(dummy)
print(f"U-Net OK — params : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ==========================================
# 7.3 WEIGHTED LOSS (Handling Class Imbalance)
# ==========================================
# Data distribution is highly skewed:
# Surface = 57.6% of pixels -> will dominate the gradient without weighting
# Cumulonimbus = 1.7% -> will be ignored by the model without weighting

# Calculate weights inversely proportional to class frequency
counts_dict = dict(zip(classes_uniq, counts))  # From analysis in section 5.1

# Initialize weights for all 9 classes (indices 0 to 8)
# Class 0 (background) is explicitly set to weight=0
class_weights = torch.zeros(9, dtype=torch.float32)
total_active_pixels = sum(counts_dict.get(c, 1) for c in range(1, 9))

for c in range(1, 9):
    # Frequency relative to the total number of non-background pixels
    frequency = counts_dict.get(c, 1) / total_active_pixels
    class_weights[c] = 1.0 / frequency

# Normalize so the sum of weights equals 1 (excluding background)
class_weights[1:] = class_weights[1:] / class_weights[1:].sum()
class_weights[0] = 0.0  # Force ignore background

print("Class Weights for Loss Function:")
for c in range(9):
    class_name = CLASS_NAMES.get(c, f'Class {c}')
    print(f"   Class {c} ({class_name:>15s}) : {class_weights[c]:.4f}")

# Define Cross-Entropy Loss with the calculated weights
criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(DEVICE),
    ignore_index=0   # Ensures background pixels do not contribute to gradient updates
)

In [ ]:
# ==========================================
# 7.4 TRAINING & VALIDATION LOOPS
# ==========================================

def compute_iou_per_class(preds, labels, n_classes=9, ignore_index=0):
    """
    Computes Intersection over Union (IoU) per class.
    IoU is the standard metric for semantic segmentation performance.
    Class 0 (background) is ignored by default.
    """
    iou_list = []
    # Iterate through classes 1 to 8
    for cls in range(1, n_classes):
        pred_mask = (preds == cls)
        label_mask = (labels == cls)
        
        # Calculate intersection and union for the specific class
        intersection = (pred_mask & label_mask).sum().item()
        union = (pred_mask | label_mask).sum().item()
        
        if union == 0:
            # Append NaN if the class is not present in the prediction or the label
            iou_list.append(float('nan'))
        else:
            iou_list.append(intersection / union)
            
    return iou_list # Returns a list of 8 IoU values (classes 1-8)


def train_one_epoch(model, loader, optimizer, criterion, device):
    """
    Executes a single training pass over the dataset.
    """
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Optimization step
        optimizer.zero_grad()
        logits = model(X_batch)               # Forward pass: shape (B, 9, 128, 128)
        loss = criterion(logits, y_batch)     # Calculate loss
        loss.backward()                        # Backward pass
        optimizer.step()                       # Update weights

        total_loss += loss.item()

    return total_loss / len(loader)


def validate(model, loader, criterion, device, n_classes=9):
    """
    Evaluates the model on the validation set.
    Computes mean loss and Mean IoU (mIoU).
    """
    model.eval()
    total_loss = 0
    all_iou_results = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item()

            # Convert logits to class predictions
            preds = torch.argmax(logits, dim=1)  # shape (B, 128, 128)
            
            # Compute batch-level IoU per class
            iou = compute_iou_per_class(preds.cpu(), y_batch.cpu(), n_classes)
            all_iou_results.append(iou)

    mean_loss = total_loss / len(loader)
    
    # Aggregate IoU values across all batches
    iou_array = np.array(all_iou_results)
    
    # Calculate Global Mean IoU (ignoring NaNs from missing classes in specific batches)
    mean_iou = np.nanmean(iou_array)
    
    # Calculate Mean IoU per class
    per_class_iou = np.nanmean(iou_array, axis=0)

    return mean_loss, mean_iou, per_class_iou

In [ ]:
# ==========================================
# 7.5 TRAINING HYPERPARAMETERS AND LOOP
# ==========================================

# Configuration
N_EPOCHS   = 30
LR         = 3e-4          # Lower learning rate used to prevent training instability
PATIENCE   = 12            # Early stopping: stops if no improvement for 12 consecutive epochs
CHECKPOINT = "./unet_best.pth"

# Optimizer and Learning Rate Scheduler
# AdamW is used with weight decay for better regularization
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
# Cosine Annealing reduces the learning rate over time to fine-tune the weights
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-6)

# Tracking metrics
history = {'train_loss': [], 'val_loss': [], 'val_miou': []}
best_miou = 0.0
epochs_no_improve = 0

print(f"Starting training on {DEVICE}...")

for epoch in tqdm(range(1, N_EPOCHS + 1)):
    # Training and Validation Phases
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_miou, iou_per_class = validate(model, val_loader, criterion, DEVICE)
    
    # Update the learning rate
    scheduler.step()

    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_miou'].append(val_miou)

    # Model Checkpointing
    # Save the model only if the validation Mean IoU improves
    if val_miou > best_miou:
        best_miou = val_miou
        epochs_no_improve = 0
        torch.save(model.state_dict(), CHECKPOINT)
        status_flag = " - Best model saved"
    else:
        # Increment early stopping counter
        epochs_no_improve += 1
        status_flag = f" - Patience: {epochs_no_improve}/{PATIENCE}"

    # Log epoch progress
    print(f"  Epoch {epoch:02d}/{N_EPOCHS}  "
          f"train_loss={train_loss:.4f}  "
          f"val_loss={val_loss:.4f}  "
          f"mIoU={val_miou:.4f}{status_flag}")

    # Early Stopping Logic
    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}. Best mIoU reached: {best_miou:.4f}")
        break

print(f"\nTraining complete. Best validation mIoU: {best_miou:.4f}")

In [ ]:
# ==========================================
# 7.6 TRAINING PLOTS
# ==========================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("U-Net GOES-16 — Courbes d'entraînement", fontweight='bold')

epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   color='tomato')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss (Cross-Entropy pondérée)")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['val_miou'], color='seagreen', lw=2)
axes[1].axhline(best_miou, color='gray', ls='--', label=f'Best mIoU = {best_miou:.4f}')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Mean IoU")
axes[1].set_title("mIoU Validation"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("unet_training.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==========================================
# 7.7 EVALUATION FINAL ON TEST SET
# ==========================================

model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.eval()

_, test_miou, iou_per_class = validate(model, test_loader, criterion, DEVICE)

print("=" * 55)
print("FINAL RESULTS \ TEST SET")
print("=" * 55)
print(f"  Mean IoU global : {test_miou:.4f}")
print()
print(f"  {'Classe':<20s}  {'IoU':>8s}")
print("  " + "-" * 32)
for cls_idx, iou in enumerate(iou_per_class, start=1):
    nom = CLASS_NAMES.get(cls_idx, f'Class {cls_idx}')
    bar = "█" * int(iou * 20) if not np.isnan(iou) else "—"
    print(f"  {nom:<20s}  {iou:>6.4f}  {bar}")

In [ ]:
# ============================================================
# COMPLETE MODEL EVALUATION
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score,
    f1_score, cohen_kappa_score, matthews_corrcoef
)

# Configuration and Class Mapping
CLASS_NAMES = {
    1: "Surface", 2: "Cumulus", 3: "Stratus", 4: "Mid-Level",
    5: "Cirrus",  6: "Cirrostratus", 7: "Snow/Ice", 8: "Cumulonimbus"
}
PALETTE = [
    "#000000", "#8B4513", "#87CEEB", "#D3D3D3", "#A9A9A9",
    "#00FFFF", "#808080", "#0000FF", "#FF00FF"
]
LABELS = list(range(1, 9))
NAMES = [CLASS_NAMES[c] for c in LABELS]

# ============================================================
# 1. PREDICTION COLLECTION ON TEST SET
# ============================================================

# Load model weights and set to evaluation mode
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch.to(DEVICE))
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        
        # Flatten tensors to 1D vectors for metric calculation
        all_preds.append(preds.reshape(-1))
        all_labels.append(y_batch.numpy().reshape(-1))

y_pred_flat = np.concatenate(all_preds)
y_true_flat = np.concatenate(all_labels)

# Filter out class 0 (background/unknown) from the evaluation
mask = y_true_flat > 0
y_pred_eval = y_pred_flat[mask]
y_true_eval = y_true_flat[mask]

# ============================================================
# 2. METRIC COMPUTATION
# ============================================================

# Intersection over Union (IoU) per class
iou_per_class = []
for cls in LABELS:
    tp = ((y_pred_eval == cls) & (y_true_eval == cls)).sum()
    fp = ((y_pred_eval == cls) & (y_true_eval != cls)).sum()
    fn = ((y_pred_eval != cls) & (y_true_eval == cls)).sum()
    union = tp + fp + fn
    iou_per_class.append(tp / union if union > 0 else float("nan"))

miou = np.nanmean(iou_per_class)

# Precision, Recall, and F1-Score per class
precision = precision_score(y_true_eval, y_pred_eval, labels=LABELS, average=None, zero_division=0)
recall = recall_score(y_true_eval, y_pred_eval, labels=LABELS, average=None, zero_division=0)
f1 = f1_score(y_true_eval, y_pred_eval, labels=LABELS, average=None, zero_division=0)

# Dice Coefficient (Equivalent to F1 for class-wise binary segmentation)
dice = f1  

# Global Averages
macro_p = precision_score(y_true_eval, y_pred_eval, labels=LABELS, average="macro", zero_division=0)
macro_r = recall_score(y_true_eval, y_pred_eval, labels=LABELS, average="macro", zero_division=0)
macro_f1 = f1_score(y_true_eval, y_pred_eval, labels=LABELS, average="macro", zero_division=0)
weighted_f1 = f1_score(y_true_eval, y_pred_eval, labels=LABELS, average="weighted", zero_division=0)
accuracy = (y_pred_eval == y_true_eval).mean()
kappa = cohen_kappa_score(y_true_eval, y_pred_eval)
mcc = matthews_corrcoef(y_true_eval, y_pred_eval)

# Pixel Accuracy per class
px_acc_per_class = []
for cls in LABELS:
    mask_cls = y_true_eval == cls
    if mask_cls.sum() == 0:
        px_acc_per_class.append(float("nan"))
    else:
        px_acc_per_class.append((y_pred_eval[mask_cls] == cls).mean())

# ============================================================
# 3. CONSOLE OUTPUT
# ============================================================

print("\n" + "=" * 72)
print(" FINAL RESULTS - TEST SET")
print("=" * 72)
header = f"{'Class':<18} {'IoU':>6} {'Dice':>6} {'Prec':>6} {'Recall':>6} {'F1':>6} {'PxAcc':>6}"
print(f"\n {header}")
print(" " + "-" * 62)

for i, (cls, name) in enumerate(zip(LABELS, NAMES)):
    iou_v = iou_per_class[i]
    print(f" {name:<18} "
          f"{iou_v:>6.4f} "
          f"{dice[i]:>6.4f} "
          f"{precision[i]:>6.4f} "
          f"{recall[i]:>6.4f} "
          f"{f1[i]:>6.4f} "
          f"{px_acc_per_class[i]:>6.4f}")

print(" " + "-" * 62)
print(f" {'MACRO AVERAGE':<18} "
      f"{miou:>6.4f} "
      f"{macro_f1:>6.4f} "
      f"{macro_p:>6.4f} "
      f"{macro_r:>6.4f} "
      f"{macro_f1:>6.4f} "
      f"{accuracy:>6.4f}")

print(f"\n Mean IoU (mIoU)    : {miou:.4f} ({miou*100:.1f}%)")
print(f" Macro-F1           : {macro_f1:.4f}")
print(f" Weighted-F1        : {weighted_f1:.4f}")
print(f" Global Accuracy    : {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f" Cohen Kappa        : {kappa:.4f}")
print(f" MCC                : {mcc:.4f}")
print("=" * 72)

# ============================================================
# 4. VISUALIZATION
# ============================================================

fig = plt.figure(figsize=(24, 18))
fig.suptitle(f"U-Net Evaluation Summary - GOES-16\n"
             f"mIoU={miou:.4f} | Macro-F1={macro_f1:.4f} | "
             f"Kappa={kappa:.4f} | Acc={accuracy:.4f}",
             fontsize=15, fontweight="bold")

# 4.1 Normalized Confusion Matrix
ax1 = fig.add_subplot(2, 3, 1)
cm = confusion_matrix(y_true_eval, y_pred_eval, labels=LABELS, normalize="true")
im = ax1.imshow(cm, cmap="Blues", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax1, label="Normalized Recall")
ax1.set_xticks(range(len(NAMES))); ax1.set_xticklabels(NAMES, rotation=40, ha="right", fontsize=8)
ax1.set_yticks(range(len(NAMES))); ax1.set_yticklabels(NAMES, fontsize=8)
ax1.set_xlabel("Predicted"); ax1.set_ylabel("Actual")
ax1.set_title("Confusion Matrix\n(Normalized by Actual Class)")

for i in range(len(NAMES)):
    for j in range(len(NAMES)):
        ax1.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                 fontsize=7, color="white" if cm[i,j] > 0.55 else "black")

# 4.2 Class-wise Metrics Bar Chart (IoU / F1 / Precision / Recall)
ax2 = fig.add_subplot(2, 3, 2)
x_pos = np.arange(len(NAMES))
bar_width = 0.2
iou_vals = [v if not np.isnan(v) else 0. for v in iou_per_class]

ax2.bar(x_pos - 1.5*bar_width, iou_vals, bar_width, label="IoU", color="steelblue", alpha=0.85)
ax2.bar(x_pos - 0.5*bar_width, precision, bar_width, label="Precision", color="seagreen", alpha=0.85)
ax2.bar(x_pos + 0.5*bar_width, recall, bar_width, label="Recall", color="tomato", alpha=0.85)
ax2.bar(x_pos + 1.5*bar_width, f1, bar_width, label="F1", color="darkorange", alpha=0.85)

ax2.axhline(miou, color="navy", ls="--", lw=1.5, label=f"mIoU={miou:.3f}")
ax2.axhline(macro_f1, color="purple", ls=":", lw=1.5, label=f"F1={macro_f1:.3f}")

ax2.set_xticks(x_pos); ax2.set_xticklabels(NAMES, rotation=40, ha="right", fontsize=8)
ax2.set_ylim(0, 1.08); ax2.set_ylabel("Score")
ax2.set_title("Per-Class Metrics")
ax2.legend(fontsize=8); ax2.grid(axis="y", alpha=0.3)

# 4.3 Dice Score Heatmap
ax3 = fig.add_subplot(2, 3, 3)
dice_vals = dice.reshape(1, -1)
im3 = ax3.imshow(dice_vals, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im3, ax=ax3, orientation="horizontal", pad=0.2, label="Dice Score")
ax3.set_xticks(range(len(NAMES))); ax3.set_xticklabels(NAMES, rotation=40, ha="right", fontsize=8)
ax3.set_yticks([]); ax3.set_title("Dice Score per Class")

for j, v in enumerate(dice):
    ax3.text(j, 0, f"{v:.3f}", ha="center", va="center",
             fontsize=10, fontweight="bold",
             color="black" if 0.3 < v < 0.8 else "white")

# 4.4 Training History (Loss + mIoU)
ax4 = fig.add_subplot(2, 3, 4)
epochs = range(1, len(history["train_loss"]) + 1)
ax4.plot(epochs, history["train_loss"], label="Train Loss", color="steelblue", lw=2)
ax4.plot(epochs, history["val_loss"], label="Val Loss", color="tomato", lw=2)
ax4_twin = ax4.twinx()
ax4_twin.plot(epochs, history["val_miou"], label="Val mIoU", color="seagreen", lw=2, ls="--")
ax4_twin.set_ylabel("mIoU", color="seagreen")
ax4_twin.tick_params(axis="y", labelcolor="seagreen")
ax4.set_xlabel("Epoch"); ax4.set_ylabel("Loss")
ax4.set_title("Training History")
h1, l1 = ax4.get_legend_handles_labels()
h2, l2 = ax4_twin.get_legend_handles_labels()
ax4.legend(h1 + h2, l1 + l2, fontsize=8)
ax4.grid(alpha=0.3)

# 4.5 Class Distribution: Ground Truth vs Prediction
ax5 = fig.add_subplot(2, 3, 5)
counts_true = [(y_true_eval == c).sum() for c in LABELS]
counts_pred = [(y_pred_eval == c).sum() for c in LABELS]
total_pixels = len(y_true_eval)
x_pos = np.arange(len(NAMES))

ax5.bar(x_pos - 0.2, [c/total_pixels*100 for c in counts_true], 0.4, label="Ground Truth", color="steelblue", alpha=0.85)
ax5.bar(x_pos + 0.2, [c/total_pixels*100 for c in counts_pred], 0.4, label="Predicted", color="tomato", alpha=0.85)
ax5.set_xticks(x_pos); ax5.set_xticklabels(NAMES, rotation=40, ha="right", fontsize=8)
ax5.set_ylabel("% Pixels"); ax5.set_title("Class Distribution Bias")
ax5.legend(fontsize=9); ax5.grid(axis="y", alpha=0.3)

# 4.6 Radar Chart (Performance Profiling)
ax6 = fig.add_subplot(2, 3, 6, polar=True)
angles = np.linspace(0, 2 * np.pi, len(NAMES), endpoint=False).tolist()
angles += angles[:1] # Close the polygon

# Repeat data to close the radar plot
iou_radar = iou_vals + iou_vals[:1]
f1_radar = list(f1) + [f1[0]]
prec_radar = list(precision) + [precision[0]]
rec_radar = list(recall) + [recall[0]]

ax6.plot(angles, iou_radar, "o-", lw=2, color="steelblue", label="IoU")
ax6.fill(angles, iou_radar, alpha=0.10, color="steelblue")
ax6.plot(angles, f1_radar, "s-", lw=2, color="darkorange", label="F1")
ax6.fill(angles, f1_radar, alpha=0.10, color="darkorange")
ax6.plot(angles, prec_radar, "^-", lw=2, color="seagreen", label="Precision")
ax6.plot(angles, rec_radar, "v-", lw=2, color="tomato", label="Recall")

ax6.set_xticks(angles[:-1])
ax6.set_xticklabels(NAMES, size=8)
ax6.set_ylim(0, 1)
ax6.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax6.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"], size=6)
ax6.set_title("Performance Profile Radar Chart", pad=15)
ax6.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)

plt.tight_layout()
plt.savefig("complete_evaluation_report.png", dpi=150, bbox_inches="tight")
plt.show()

print("Evaluation report saved to: complete_evaluation_report.png")

In [ ]:
# ==========================================
# 7.8 PREDICTION VISUALIZATION
# ==========================================

# Definition of the class color palette for segmentation masks
class_colors = [
    '#000000',  # 0: Background
    '#8B4513',  # 1: Surface
    '#87CEEB',  # 2: Cumulus
    '#D3D3D3',  # 3: Stratus
    '#A9A9A9',  # 4: Mid-Level
    '#00FFFF',  # 5: Cirrus
    '#FFFFFF',  # 6: Cirrostratus
    '#0000FF',  # 7: Snow/Ice
    '#FF00FF',  # 8: Cumulonimbus
]

# Create colormap and normalization boundaries for categorical plotting
cmap_expert = ListedColormap(class_colors)
norm_expert = BoundaryNorm(np.arange(-0.5, 9.5, 1), cmap_expert.N)

def visualize_predictions(model, dataset, n_examples=4, device=DEVICE):
    """
    Displays n_examples of patches side-by-side:
    Raw Thermal IR (C13) | Expert Ground Truth | U-Net Prediction
    """
    model.eval()
    
    # Select random samples from the dataset
    indices = np.random.choice(len(dataset), n_examples, replace=False)

    fig, axes = plt.subplots(n_examples, 3, figsize=(12, n_examples * 4))
    fig.suptitle("U-Net GOES-16 - Results on Test Set\n"
                 "C13 Thermal IR | Ground Truth | U-Net Prediction",
                 fontsize=13, fontweight='bold')

    for row, idx in enumerate(indices):
        x_tensor, y_true_tensor = dataset[idx]

        # Generate model prediction
        with torch.no_grad():
            # Add batch dimension and move to device
            input_batch = x_tensor.unsqueeze(0).to(device)
            logits = model(input_batch)  # Output shape: (1, 9, 128, 128)
            y_pred = torch.argmax(logits, dim=1).squeeze().cpu().numpy()

        # Retrieve C13 channel (located at index 0 of our optimized feature stack)
        c13_data = x_tensor[0].numpy()

        # Column 1: Raw Thermal Imagery
        axes[row, 0].imshow(c13_data, cmap='gray_r')
        axes[row, 0].set_title(f"Thermal C13 - Patch #{idx}")
        axes[row, 0].axis('off')

        # Column 2: Expert Ground Truth
        axes[row, 1].imshow(y_true_tensor.numpy(), cmap=cmap_expert, norm=norm_expert)
        axes[row, 1].set_title("Ground Truth (Expert Tree)")
        axes[row, 1].axis('off')

        # Column 3: Model Prediction
        axes[row, 2].imshow(y_pred, cmap=cmap_expert, norm=norm_expert)
        axes[row, 2].set_title("U-Net Prediction")
        axes[row, 2].axis('off')

    # Construct a shared legend below the plots
    legend_patches = [
        mpatches.Patch(color=class_colors[c], label=CLASS_NAMES[c])
        for c in range(1, 9)
    ]
    
    fig.legend(handles=legend_patches, loc='lower center',
               ncol=4, fontsize=9, framealpha=0.9,
               bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout()
    plt.savefig("unet_prediction_visuals.png", dpi=150, bbox_inches='tight')
    plt.show()

# Execute visualization on the test set
visualize_predictions(model, test_ds, n_examples=4)

In [ ]:
# ==========================================
# 7.10 FULL ZONE RECONSTRUCTION
# ==========================================

import torch
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

def predict_full_zone(test_date, model, zone_coords, expert_func, device=DEVICE):
    """
    Downloads a complete scene and reconstructs the classification map
    by stitching together U-Net patch predictions.
    """
    # Define required bands and target area
    bands = ['C01','C02','C03','C04','C05','C06','C07','C08','C09','C10','C13','C15']
    area_def = get_area_definition(zone_coords)

    # 1. Load and Resample Raw Data
    print(f"Loading scene data for: {test_date.strftime('%H:%M UTC')}...")
    scn_raw = load_sat_data(test_date, bands)
    scn_proj = resample_with_cache(scn_raw, area_def)

    # 2. Expert Ground Truth Generation (Expert Tree)
    # Generate reference masks for the entire projected scene
    masks_expert = expert_func(scn_proj)
    H, W = scn_proj['C13'].values.shape
    y_expert_full = np.zeros((H, W), dtype=np.uint8)

    # Populate the expert classification map
    class_map = {
        'Surface': 1, 'Cumulus': 2, 'Stratus': 3, 'Mid-Level': 4,
        'Cirrus': 5, 'Cirrostratus': 6, 'Snow/Ice': 7, 'Cumulonimbus': 8
    }
    for name, mask in masks_expert:
        for key, idx in class_map.items():
            if key in name:
                y_expert_full[mask] = idx

    # 3. Feature Preparation for U-Net (C13, C07, C04)
    # Stack the same bands used during the training phase
    X_img = np.stack([
        scn_proj['C13'].values,
        scn_proj['C07'].values,
        scn_proj['C04'].values
    ], axis=-1).astype(np.float32)
    X_img = np.nan_to_num(X_img)

    # Fixed Normalization
    # Critical for maintaining temporal consistency in predictions.
    # Replace these with your specific training min/max values if necessary.
    MIN_VALS = np.array([200.0, 200.0, 0.0])
    MAX_VALS = np.array([320.0, 320.0, 1.0])
    
    for i in range(3):
        denominator = MAX_VALS[i] - MIN_VALS[i] + 1e-8
        X_img[..., i] = (X_img[..., i] - MIN_VALS[i]) / denominator
    X_img = np.clip(X_img, 0, 1)

    # 4. Block-based Inference (Tiling)
    print("Executing U-Net inference across the zone...")
    y_pred_full = np.zeros((H, W), dtype=np.uint8)
    PATCH_SIZE = 128

    model.eval()
    with torch.no_grad():
        # Iterate through the image using 128x128 patches
        for i in range(0, H - PATCH_SIZE + 1, PATCH_SIZE):
            for j in range(0, W - PATCH_SIZE + 1, PATCH_SIZE):
                # Extract patch
                patch_x = X_img[i:i+PATCH_SIZE, j:j+PATCH_SIZE, :]
                
                # Convert to PyTorch tensor (C, H, W) and add batch dimension
                input_t = torch.from_numpy(patch_x).permute(2, 0, 1).unsqueeze(0).to(device)

                # Predict and update the full map
                logits = model(input_t)
                preds = torch.argmax(logits, dim=1).squeeze().cpu().numpy()
                y_pred_full[i:i+PATCH_SIZE, j:j+PATCH_SIZE] = preds

    # 5. Comparative Visualization
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    # Subplot 1: Thermal IR Band
    axes[0].imshow(scn_proj['C13'].values, cmap='gray_r')
    axes[0].set_title(f"Thermal IR (C13) - {test_date.strftime('%H:%M')}")

    # Subplot 2: Expert Ground Truth
    axes[1].imshow(y_expert_full, cmap=cmap_expert, norm=norm_expert)
    axes[1].set_title("Expert Tree (Ground Truth)")

    # Subplot 3: U-Net Prediction
    axes[2].imshow(y_pred_full, cmap=cmap_expert, norm=norm_expert)
    axes[2].set_title("Reconstructed U-Net Prediction")

    for ax in axes:
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

# Example execution for a specific timestamp
# Define the date, coordinates, and expert function before calling
test_timestamp = datetime(2023, 7, 22, 18, 0)
predict_full_zone(test_timestamp, model, ma_zone, get_expert_masks_final)